# EDA contentieux : recherche du périmètre idéal de la population CTX

**Objectif** : définir une règle métier qui détecte et isole les clients en gestion « contentieuse » (codification PAY_n = 2 figée, faux rétrogradages), afin de :
- les sortir du périmètre du ML et les traiter par une règle métier (prédits en défaut),
- entraîner les modèles sur une population au comportement homogène.

**Méthodologie**
- Les règles candidates sont définies **a priori, par logique métier**, à partir des constats de l'EDA lab. Aucun seuil n'est optimisé sur la cible.
- Le jeu est découpé **avant toute analyse** en train / test (80/20, stratifié sur `dpnm`, `random_state=42`). Toutes les analyses sont faites sur le **train**.
- Les règles sont d'abord décrites par leur **comportement** (volumes, recouvrements, paiements, dette), sans `dpnm`.
- `dpnm` n'est utilisé sur le train que pour **valider** qu'un périmètre a un sens métier. Le **test** n'est utilisé **qu'une seule fois**, à la fin, pour la règle retenue.

**Données** : `cleaned3_creditcard.csv` (niveau de correction 3 v2), restreint au périmètre ML `S12` (`BILL_AMT1 > 0` et `LIMIT_BAL <= 500 000`).
La population contentieuse est quasi inchangée entre les niveaux de correction 0 et 3 : aucune correction ne crée ni ne modifie de codes PAY_n >= 2.

## 1. Chargement et périmètre

In [53]:
import pandas as pd
import numpy as np
from pathlib import Path

# Retrouve la racine du projet (là où se trouve le dossier 'data')
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "cleaned3_creditcard.csv")
print(f"Dataset complet : {len(df)} lignes | taux de défaut : {df['dpnm'].mean() * 100:.2f}%")

# Périmètre ML S12 : encours positif à M-1 et plafond <= 500 000 NT$
df = df[(df['BILL_AMT1'] > 0) & (df['LIMIT_BAL'] <= 500000)].copy()
print(f"Périmètre S12   : {len(df)} lignes | taux de défaut : {df['dpnm'].mean() * 100:.2f}%")

pay_cols = [f'PAY_{i}' for i in range(1, 7)]
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
pay_amt_cols = [f'PAY_AMT{i}' for i in range(1, 7)]

Dataset complet : 29136 lignes | taux de défaut : 21.69%
Périmètre S12   : 27202 lignes | taux de défaut : 21.95%


## 2. Séparation train / test
Le split est fait avant toute analyse et ne dépend d'aucune définition du CTX. Les notebooks ML devront reproduire ce même split (même périmètre, `stratify=dpnm`, `random_state=42`).

In [54]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df, test_size=0.2, stratify=df['dpnm'], random_state=42)

print(f"Train : {len(df_train)} lignes | taux de défaut : {df_train['dpnm'].mean() * 100:.2f}%")
print(f"Test  : {len(df_test)} lignes | taux de défaut : {df_test['dpnm'].mean() * 100:.2f}%")

Train : 21761 lignes | taux de défaut : 21.95%
Test  : 5441 lignes | taux de défaut : 21.96%


## 3. Règles candidates

Rappel des constats de l'EDA lab (dataset complet, avant filtrage) :
- 530 clients avec PAY_n = 2 sur les 6 mois (1.77 %) : la codification ne bouge pas, que le client paie ou non, et la dette augmente dans 87 % des mois sans paiement
- 393 « faux rétrogradages » (1.31 %) : passage de PAY_(n+1) > 2 à PAY_n = 2 sans aucun paiement et avec une dette positive
- 1 193 clients avec au moins un PAY_n > 2 (3.98 %)

Règles candidates, chacune justifiée par une définition métier :

| Règle | Définition | Logique métier |
|---|---|---|
| `R1_chronique` | PAY_n = 2 sur les 6 mois | codification figée, gestion hors circuit normal |
| `R2_faux_retrogradage` | ∃n : PAY_n = 2, PAY_(n+1) > 2, BILL_AMTn > 0, PAY_AMTn = 0 | amélioration du code sans paiement : le code 2 n'est plus un retard réel |
| `R3_actuelle` | R1 ou R2 | règle utilisée dans `S12_7` |
| `R4_retard_6m` | PAY_n >= 2 sur les 6 mois | retard sur toute la période observée (inclut R1) |
| `R5_retard_profond` | au moins un PAY_n > 2 sur les 6 mois | un seul retard d'au moins 3 mois suffit (inclut R2, qui exige un PAY_(n+1) > 2) |
| `R6_retard_continu` | PAY_n >= 2 sans interruption de M-1 jusqu'à M-k, avec 2 <= k <= 5 | retard installé depuis au moins 2 mois et toujours en cours à M-1 (6 mois = R4) |
| `R7_R1_R2_R4` | R1 ou R2 ou R4 | retard figé ou permanent sur la période, ou faux rétrogradage (R1 étant inclus dans R4, équivaut à R2 ou R4) |
| `R8_elargie` | R1 ou R2 ou R4 ou R5 ou R6 | toutes les situations de retard profond, figé ou installé |

**Règles bis (`_bis`)** : PAY_1 = 1, PAY_2 >= 2 et même règle appliquée sur la fenêtre PAY_2 à PAY_6.  
Les règles R1, R4 et R6 exigent un retard à M-1 et excluent donc mécaniquement les PAY_1 = 1, alors qu'un passage de >= 2 à 1 après un contentieux ne signifie pas forcément une sortie.  
Le paiement en M-1 ou en M-2 (le paiement ayant fait passer le code à 1 peut être enregistré le mois précédent) est analysé via le statut à M-1 (section 3.1).  
Pour `R6_retard_continu_bis`, la série continue part de M-2 (2 à 4 mois, 5 mois = `R4_retard_6m_bis`).

In [55]:
# Les règles s'appliquent sur une fenêtre de mois :
# - règles de base : PAY_1 à PAY_6
# - règles bis : PAY_1 == 1, PAY_2 >= 2 et même règle appliquée sur PAY_2 à PAY_6
mois_base = [1, 2, 3, 4, 5, 6]
mois_bis = [2, 3, 4, 5, 6]

def cols(mois):
    return [f'PAY_{n}' for n in mois]

def regle_R1(d, mois):
    return (d[cols(mois)] == 2).all(axis=1)

def regle_R2(d, mois):
    masque = pd.Series(False, index=d.index)
    for n in mois[:-1]:
        masque |= (
            (d[f'PAY_{n}'] == 2)
            & (d[f'PAY_{n+1}'] > 2)
            & (d[f'BILL_AMT{n}'] > 0)
            & (d[f'PAY_AMT{n}'] == 0)
        )
    return masque

def regle_R4(d, mois):
    return (d[cols(mois)] >= 2).all(axis=1)

def regle_R5(d, mois):
    return (d[cols(mois)] > 2).any(axis=1)

def regle_R6(d, mois):
    # Durée de la série continue de PAY_n >= 2 en partant du mois le plus récent de la fenêtre
    duree_retard_continu = (d[cols(mois)] >= 2).cumprod(axis=1).sum(axis=1)
    # Au moins 2 mois, fenêtre complète exclue (= R4)
    return duree_retard_continu.between(2, len(mois) - 1)

regles_base = {
    'R1_chronique': regle_R1,
    'R2_faux_retrogradage': regle_R2,
    'R3_actuelle': lambda d, mois: regle_R1(d, mois) | regle_R2(d, mois),
    'R4_retard_6m': regle_R4,
    'R5_retard_profond': regle_R5,
    'R6_retard_continu': regle_R6,
    'R7_R1_R2_R4': lambda d, mois: regle_R1(d, mois) | regle_R2(d, mois) | regle_R4(d, mois),
    'R8_elargie': lambda d, mois: (
        regle_R1(d, mois) | regle_R2(d, mois) | regle_R4(d, mois) | regle_R5(d, mois) | regle_R6(d, mois)
    ),
}

def condition_bis(d):
    return (d['PAY_1'] == 1) & (d['PAY_2'] >= 2)

regles = {}
for nom, f in regles_base.items():
    regles[nom] = lambda d, f=f: f(d, mois_base)
for nom, f in regles_base.items():
    regles[f'{nom}_bis'] = lambda d, f=f: condition_bis(d) & f(d, mois_bis)

masques_train = pd.DataFrame({nom: f(df_train) for nom, f in regles.items()})

### 3.1 Statut CTX à M-1 : maintien ou sortie

Les règles détectent un **passage** au CTX sur la période, mais le client peut en être sorti depuis. Chaque client ayant au moins un PAY_n >= 2 reçoit un statut à M-1, à partir du **dernier mois où PAY_n >= 2** :

| Statut | Définition | Traitement envisagé |
|---|---|---|
| `Maintien M-1` | PAY_1 >= 2 : le client est toujours en retard à M-1 | règle métier, prédit en défaut |
| `PAY_1=1 paiement M-1 ou M-2` | PAY_2 >= 2 puis PAY_1 = 1, avec un paiement en M-1 ou en M-2 (PAY_AMT1 > 0 ou PAY_AMT2 > 0) | le passage à 1 s'explique par un paiement, parfois enregistré le mois précédent |
| `PAY_1=1 sans paiement M-1 ni M-2` | PAY_2 >= 2 puis PAY_1 = 1, sans aucun paiement en M-1 ni en M-2 | passage à 1 inexpliqué : sortie non prouvée |
| `Sortie` | autres cas avec PAY_1 < 2 et au moins un paiement visible (PAY_AMTn > 0) depuis le dernier mois à >= 2 | retour au ML avec un flag d'antériorité CTX |
| `Indéterminé` | autres cas avec PAY_1 < 2 sans aucun paiement visible depuis le dernier mois à >= 2 | sortie non prouvée : à trancher |
| `Jamais >= 2` | aucun PAY_n >= 2 sur les 6 mois | hors sujet |

In [56]:
def statut_ctx(d):
    retard = (d[pay_cols] >= 2).values
    a_eu_retard = retard.any(axis=1)
    # Position du dernier mois à >= 2 (0 = M-1, 5 = M-6)
    rang_dernier_retard = retard.argmax(axis=1)
    # Mois plus récents que le dernier mois à >= 2
    mois_depuis = np.arange(6) < rang_dernier_retard[:, None]
    paiement_depuis = ((d[pay_amt_cols].values > 0) & mois_depuis).any(axis=1)
    # Passage de PAY_2 = 2 à PAY_1 = 1 : paiement en M-1 ou en M-2
    passage_2_vers_1 = condition_bis(d).values
    paiement_m1_m2 = (d['PAY_AMT1'].values > 0) | (d['PAY_AMT2'].values > 0)

    statut = np.select(
        [
            ~a_eu_retard,
            rang_dernier_retard == 0,
            passage_2_vers_1 & paiement_m1_m2,
            passage_2_vers_1,
            paiement_depuis,
        ],
        ['Jamais >= 2', 'Maintien M-1', 'PAY_1=1 paiement M-1 ou M-2', 'PAY_1=1 sans paiement M-1 ni M-2', 'Sortie'],
        default='Indéterminé',
    )
    return pd.Series(statut, index=d.index)

statuts = ['Maintien M-1', 'PAY_1=1 paiement M-1 ou M-2', 'PAY_1=1 sans paiement M-1 ni M-2', 'Sortie', 'Indéterminé']
statut_train = statut_ctx(df_train)
print(statut_train.value_counts())

Jamais >= 2                         15193
Maintien M-1                         2486
Sortie                               2326
PAY_1=1 paiement M-1 ou M-2          1390
Indéterminé                           282
PAY_1=1 sans paiement M-1 ni M-2       84
Name: count, dtype: int64


## 4. Description des périmètres (sans la cible)
### 4.1 Volumes et recouvrements

In [57]:
volumes = pd.DataFrame({
    'Nb clients': masques_train.sum(),
    'Part train (%)': (masques_train.mean() * 100).round(2),
}).rename_axis('Règle')
display(volumes)

# Recouvrement : part des clients de la règle en ligne également captés par la règle en colonne
recouvrement = pd.DataFrame(
    {c: [(masques_train[l] & masques_train[c]).sum() / max(masques_train[l].sum(), 1) * 100 for l in regles] for c in regles},
    index=list(regles),
).round(1).rename_axis('Règle (ligne) captée par ->')
print("Recouvrement (% des clients de la ligne captés par la colonne) :")
display(recouvrement)

,Nb clients,Part train (%)
Règle,,
R1_chronique,424,1.95
R2_faux_retrogradage,318,1.46
R3_actuelle,742,3.41
R4_retard_6m,756,3.47
R5_retard_profond,949,4.36
R6_retard_continu,951,4.37
R7_R1_R2_R4,896,4.12
R8_elargie,2088,9.60
R1_chronique_bis,232,1.07


Recouvrement (% des clients de la ligne captés par la colonne) :


,R1_chronique,R2_faux_retrogradage,R3_actuelle,R4_retard_6m,R5_retard_profond,R6_retard_continu,R7_R1_R2_R4,R8_elargie,R1_chronique_bis,R2_faux_retrogradage_bis,R3_actuelle_bis,R4_retard_6m_bis,R5_retard_profond_bis,R6_retard_continu_bis,R7_R1_R2_R4_bis,R8_elargie_bis
Règle (ligne) captée par ->,,,,,,,,,,,,,,,,
R1_chronique,100.0,0.0,100.0,100.0,0.0,0.0,100.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R2_faux_retrogradage,0.0,100.0,100.0,56.0,100.0,11.0,100.0,100.0,0.0,21.4,21.4,15.4,21.4,5.7,21.4,21.4
R3_actuelle,57.1,42.9,100.0,81.1,42.9,4.7,100.0,100.0,0.0,9.2,9.2,6.6,9.2,2.4,9.2,9.2
R4_retard_6m,56.1,23.5,79.6,100.0,43.9,0.0,100.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R5_retard_profond,0.0,33.5,33.5,35.0,100.0,24.9,49.7,100.0,0.0,7.2,7.2,11.2,21.2,8.7,13.2,21.2
R6_retard_continu,0.0,3.7,3.7,0.0,24.8,100.0,3.7,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R7_R1_R2_R4,47.3,35.5,82.8,84.4,52.7,3.9,100.0,100.0,0.0,7.6,7.6,5.5,7.6,2.0,7.6,7.6
R8_elargie,20.3,15.2,35.5,36.2,45.5,45.5,42.9,100.0,0.0,3.3,3.3,5.1,9.6,4.0,6.0,9.6
R1_chronique_bis,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,100.0,100.0,0.0,0.0,100.0,100.0


In [58]:
# Ventilation de chaque règle par statut à M-1
ventilation_statut = pd.DataFrame(
    {s: [(masques_train[nom] & (statut_train == s)).sum() for nom in regles] for s in statuts},
    index=list(regles),
).rename_axis('Règle')
print("Ventilation des clients de chaque règle par statut à M-1 :")
display(ventilation_statut)

Ventilation des clients de chaque règle par statut à M-1 :


,Maintien M-1,PAY_1=1 paiement M-1 ou M-2,PAY_1=1 sans paiement M-1 ni M-2,Sortie,Indéterminé
Règle,,,,,
R1_chronique,424,0,0,0,0
R2_faux_retrogradage,215,65,3,33,2
R3_actuelle,639,65,3,33,2
R4_retard_6m,756,0,0,0,0
R5_retard_profond,579,140,61,125,44
R6_retard_continu,951,0,0,0,0
R7_R1_R2_R4,793,65,3,33,2
R8_elargie,1718,140,61,125,44
R1_chronique_bis,0,231,1,0,0


### 4.2 Comportement de paiement et de dette

In [59]:
def profil_comportement(d):
    # Nombre de mois sans aucun paiement
    mois_sans_paiement = (d[pay_amt_cols] == 0).sum(axis=1)
    # Évolution de la dette de M-6 à M-1
    evolution_dette = d['BILL_AMT1'] - d['BILL_AMT6']
    # Utilisation du plafond à M-1
    utilisation = d['BILL_AMT1'] / d['LIMIT_BAL'] * 100
    return pd.Series({
        'Nb clients': len(d),
        'Mois sans paiement (moy.)': mois_sans_paiement.mean(),
        '% avec 6 mois sans paiement': (mois_sans_paiement == 6).mean() * 100,
        '% dette en hausse M-6 -> M-1': (evolution_dette > 0).mean() * 100,
        'Utilisation plafond M-1 (médiane %)': utilisation.median(),
        'LIMIT_BAL (médiane)': d['LIMIT_BAL'].median(),
    })

profils = {nom: profil_comportement(df_train[masques_train[nom]]) for nom in regles}
tous_ctx = masques_train.any(axis=1)
profils['Hors toutes règles'] = profil_comportement(df_train[~tous_ctx])
display(pd.DataFrame(profils).T.round(1).rename_axis('Règle'))

,Nb clients,Mois sans paiement (moy.),% avec 6 mois sans paiement,% dette en hausse M-6 -> M-1,Utilisation plafond M-1 (médiane %),LIMIT_BAL (médiane)
Règle,,,,,,
R1_chronique,424.0,1.4,2.8,13.7,66.6,90000.0
R2_faux_retrogradage,318.0,3.7,27.4,17.3,47.7,30000.0
R3_actuelle,742.0,2.4,13.3,15.2,60.7,60000.0
R4_retard_6m,756.0,2.4,16.1,17.5,63.8,70000.0
R5_retard_profond,949.0,2.9,13.0,36.6,66.4,50000.0
R6_retard_continu,951.0,1.7,3.6,57.2,80.9,70000.0
R7_R1_R2_R4,896.0,2.4,13.6,17.5,63.8,60000.0
R8_elargie,2088.0,2.1,7.8,38.5,70.1,60000.0
R1_chronique_bis,232.0,2.0,0.4,12.5,58.9,70000.0


In [60]:
# Comportement par statut à M-1, sur l'ensemble des clients ayant eu au moins un PAY_n >= 2
profils_statut = {s: profil_comportement(df_train[statut_train == s]) for s in statuts}
profils_statut['Jamais >= 2'] = profil_comportement(df_train[statut_train == 'Jamais >= 2'])
display(pd.DataFrame(profils_statut).T.round(1).rename_axis('Statut à M-1'))

,Nb clients,Mois sans paiement (moy.),% avec 6 mois sans paiement,% dette en hausse M-6 -> M-1,Utilisation plafond M-1 (médiane %),LIMIT_BAL (médiane)
Statut à M-1,,,,,,
Maintien M-1,2486.0,1.6,6.3,44.6,72.6,80000.0
PAY_1=1 paiement M-1 ou M-2,1390.0,1.7,0.0,48.8,69.7,60000.0
PAY_1=1 sans paiement M-1 ni M-2,84.0,4.1,23.8,78.6,65.5,30000.0
Sortie,2326.0,1.3,0.0,43.9,45.2,80000.0
Indéterminé,282.0,2.9,7.1,66.0,1.6,125000.0
Jamais >= 2,15193.0,0.7,1.7,67.0,29.8,160000.0


**Interprétation** : *à compléter*

## 5. Validation sur le train avec la cible
Pour chaque périmètre :
- **taux de défaut** dans le périmètre (précision de la règle « tous en défaut »),
- **part des défauts captés dans tout le jeu de données** (recall de la règle),
- **taux de défaut du reste du jeu de données** : population laissée au ML.

In [61]:
def validation(d, masque):
    y = d['dpnm']
    return pd.Series({
        'Nb clients': masque.sum(),
        'Part (%)': masque.mean() * 100,
        'Taux défaut périmètre (%)': y[masque].mean() * 100,
        'Défauts captés (%)': y[masque].sum() / y.sum() * 100,
        'Taux défaut reste (%)': y[~masque].mean() * 100,
    })

print(f"Taux de défaut global (train) : {df_train['dpnm'].mean() * 100:.2f}%")
display(pd.DataFrame({nom: validation(df_train, masques_train[nom]) for nom in regles}).T.round(2).rename_axis('Règle'))

Taux de défaut global (train) : 21.95%


,Nb clients,Part (%),Taux défaut périmètre (%),Défauts captés (%),Taux défaut reste (%)
Règle,,,,,
R1_chronique,424.0,1.95,79.01,7.01,20.82
R2_faux_retrogradage,318.0,1.46,67.61,4.50,21.28
R3_actuelle,742.0,3.41,74.12,11.51,20.11
R4_retard_6m,756.0,3.47,78.04,12.35,19.93
R5_retard_profond,949.0,4.36,62.49,12.41,20.10
R6_retard_continu,951.0,4.37,65.40,13.02,19.97
R7_R1_R2_R4,896.0,4.12,74.44,13.96,19.70
R8_elargie,2088.0,9.60,67.39,29.45,17.13
R1_chronique_bis,232.0,1.07,53.88,2.62,21.61


In [62]:
# Taux de défaut de chaque règle ventilé par statut à M-1
taux_statut = pd.DataFrame(
    {s: [df_train.loc[masques_train[nom] & (statut_train == s), 'dpnm'].mean() * 100 for nom in regles] for s in statuts},
    index=list(regles),
).round(2).rename_axis('Règle')
print("Taux de défaut (%) des clients de chaque règle, par statut à M-1 :")
display(taux_statut)

print("Validation par statut, tous clients ayant eu au moins un PAY_n >= 2 :")
display(pd.DataFrame({s: validation(df_train, statut_train == s) for s in statuts}).T.round(2).rename_axis('Statut à M-1'))

Taux de défaut (%) des clients de chaque règle, par statut à M-1 :


,Maintien M-1,PAY_1=1 paiement M-1 ou M-2,PAY_1=1 sans paiement M-1 ni M-2,Sortie,Indéterminé
Règle,,,,,
R1_chronique,79.01,NaN,NaN,NaN,NaN
R2_faux_retrogradage,73.02,56.92,100.00,51.52,50.00
R3_actuelle,77.00,56.92,100.00,51.52,50.00
R4_retard_6m,78.04,NaN,NaN,NaN,NaN
R5_retard_profond,70.12,58.57,49.18,40.00,56.82
R6_retard_continu,65.40,NaN,NaN,NaN,NaN
R7_R1_R2_R4,76.80,56.92,100.00,51.52,50.00
R8_elargie,71.01,58.57,49.18,40.00,56.82
R1_chronique_bis,NaN,53.68,100.00,NaN,NaN


Validation par statut, tous clients ayant eu au moins un PAY_n >= 2 :


,Nb clients,Part (%),Taux défaut périmètre (%),Défauts captés (%),Taux défaut reste (%)
Statut à M-1,,,,,
Maintien M-1,2486.0,11.42,69.15,35.98,15.87
PAY_1=1 paiement M-1 ou M-2,1390.0,6.39,41.51,12.08,20.62
PAY_1=1 sans paiement M-1 ni M-2,84.0,0.39,50.00,0.88,21.84
Sortie,2326.0,10.69,25.24,12.29,21.56
Indéterminé,282.0,1.30,38.30,2.26,21.74


**Interprétation** : *à compléter*

## 6. Choix du périmètre
*Justification métier du choix : à compléter*

La règle retenue est renseignée dans la cellule ci-dessous.

In [63]:
regle_retenue = 'R7_R1_R2_R4'

## 7. Validation unique sur le test
Cette section n'est exécutée qu'une fois la règle choisie : le test ne doit pas servir à comparer les règles.

In [64]:
masque_test = regles[regle_retenue](df_test)
masque_train = masques_train[regle_retenue]

display(pd.DataFrame({
    'Train': validation(df_train, masque_train),
    'Test': validation(df_test, masque_test),
}).T.round(2))

,Nb clients,Part (%),Taux défaut périmètre (%),Défauts captés (%),Taux défaut reste (%)
Train,896.0,4.12,74.44,13.96,19.70
Test,218.0,4.01,72.94,13.31,19.84


## 8. Règle finale à reporter dans les notebooks ML
Fonction à recopier telle quelle dans les notebooks ML pour créer le flag `CTX`.

In [65]:
def flag_ctx(d):
    return regles[regle_retenue](d)

print(f"Règle retenue : {regle_retenue}")
print(f"Clients CTX sur le périmètre S12 : {flag_ctx(df).sum()} ({flag_ctx(df).mean() * 100:.2f}%)")

Règle retenue : R7_R1_R2_R4
Clients CTX sur le périmètre S12 : 1114 (4.10%)
